In [ ]:
import json
import os
from pathlib import Path

workspace = Path(os.getenv("GITHUB_WORKSPACE") or Path.cwd()).resolve()
original_path = workspace / "Part_4.ipynb"
if not original_path.is_file():
    raise FileNotFoundError("Keep the original Part_4.ipynb beside Part_4_v2.ipynb.")

original_nb = json.loads(original_path.read_text(encoding="utf-8"))
code_cells = [cell for cell in original_nb.get("cells", []) if cell.get("cell_type") == "code"]
if len(code_cells) != 1:
    raise RuntimeError(f"Expected one code cell in original Part_4.ipynb, found {len(code_cells)}.")

raw_source = code_cells[0].get("source", "")
source = "".join(raw_source) if isinstance(raw_source, list) else str(raw_source)

def replace_once(old, new, label):
    global source
    count = source.count(old)
    if count != 1:
        raise RuntimeError(f"Safe patch stopped: expected one {label} anchor, found {count}.")
    source = source.replace(old, new, 1)

def replace_section(start_marker, end_marker, replacement, label):
    global source
    if source.count(start_marker) != 1 or source.count(end_marker) != 1:
        raise RuntimeError(f"Safe patch stopped: {label} section markers were not unique.")
    start = source.index(start_marker)
    end = source.index(end_marker, start)
    if end <= start:
        raise RuntimeError(f"Safe patch stopped: invalid {label} section boundaries.")
    source = source[:start] + replacement.rstrip() + "\n\n" + source[end:]

replace_once(
    '    "Generated At"\n]',
    '    "Generated At",\n'
    '    "Metadata Generation Status",\n'
    '    "Discord Upload Status",\n'
    '    "Discord Attachment URL",\n'
    '    "Discord Message URL",\n'
    '    "Discord Message ID",\n'
    '    "Discord Channel ID",\n'
    '    "Discord File Size MB",\n'
    '    "Discord Error"\n]',
    "metadata header"
)

replace_section(
    "OPENROUTER_MODELS = [",
    "\n\nSHEET_HEADER = [",
    """OPENROUTER_MODELS = ["openrouter/free"]

GROQ_MODELS = [
    "openai/gpt-oss-120b",
    "openai/gpt-oss-20b",
    "qwen/qwen3.8-27b",
]""",
    "AI model configuration"
)

openrouter_function = r"""
def openrouter_generate(prompt):
    if not OPENROUTER_API_KEY:
        raise RuntimeError("OpenRouter API key is missing")

    model_candidate = OPENROUTER_MODELS[0]
    last_exception = None

    for attempt in range(1, 3):
        response = None
        retryable = True
        try:
            response = requests.post(
                OPENROUTER_URL,
                headers={
                    "Authorization": f"Bearer {OPENROUTER_API_KEY}",
                    "Content-Type": "application/json",
                    "HTTP-Referer": "https://github.com/",
                    "X-Title": "Social Media Metadata Ranking Engine",
                },
                json={
                    "model": model_candidate,
                    "messages": [
                        {
                            "role": "system",
                            "content": (
                                "You are an expert social-media metadata writer. "
                                "Return one valid JSON object only. Do not fabricate "
                                "facts, people, news, or current trends."
                            ),
                        },
                        {"role": "user", "content": prompt},
                    ],
                    "temperature": 0.4,
                    "max_tokens": 2200,
                    "response_format": {"type": "json_object"},
                    "stream": False,
                },
                timeout=(15, 90),
            )

            if not response.ok:
                retryable = (
                    response.status_code in (408, 429)
                    or 500 <= response.status_code < 600
                )
                raise RuntimeError(
                    f"OpenRouter HTTP {response.status_code}: {response.text[:500]}"
                )

            payload = response.json()
            choices = payload.get("choices") or []
            if not choices:
                raise ValueError("OpenRouter returned no choices.")

            message = choices[0].get("message") or {}
            content = message.get("content", "")
            if isinstance(content, list):
                content = "".join(
                    str(part.get("text", "")) if isinstance(part, dict) else str(part)
                    for part in content
                )
            if not str(content).strip():
                raise ValueError("OpenRouter returned an empty message.")

            actual_model = str(payload.get("model") or model_candidate)
            print(f"    OpenRouter model selected: {actual_model}")
            return normalize_metadata(extract_json(content)), actual_model

        except Exception as exc:
            last_exception = exc
            print(f"    OpenRouter attempt {attempt}/2 failed: {str(exc)[:400]}")
            if not retryable or attempt == 2:
                break

            delay = 2.0
            if response is not None and response.status_code == 429:
                try:
                    delay = float(response.headers.get("Retry-After", "2"))
                except (TypeError, ValueError):
                    delay = 2.0
            time.sleep(min(max(delay, 1.0), 30.0))

    raise last_exception or RuntimeError("OpenRouter free-model routing failed.")
"""
replace_section(
    "def openrouter_generate(prompt):",
    "\n\n# =============================================================================\n# 8. GROQ GENERATION",
    openrouter_function,
    "OpenRouter generation"
)

groq_function = r"""
_GROQ_MODEL_CANDIDATES_CACHE = None

def _groq_model_candidates():
    global _GROQ_MODEL_CANDIDATES_CACHE
    if _GROQ_MODEL_CANDIDATES_CACHE is not None:
        return list(_GROQ_MODEL_CANDIDATES_CACHE)

    configured_model = os.getenv("GROQ_MODEL", "").strip()
    preferred = ([configured_model] if configured_model else []) + list(GROQ_MODELS)
    preferred = list(dict.fromkeys(model for model in preferred if model))

    try:
        catalog = groq_client.models.list()
        entries = getattr(catalog, "data", []) or []
        available = set()

        for item in entries:
            model_id = item.get("id") if isinstance(item, dict) else getattr(item, "id", None)
            if model_id:
                available.add(str(model_id))

        if available:
            selected = [model for model in preferred if model in available]

            if not selected:
                excluded = (
                    "whisper", "canopylabs", "guard", "embedding",
                    "rerank", "text-to-speech", "tts",
                )
                selected = sorted(
                    model for model in available
                    if not any(token in model.lower() for token in excluded)
                )[:4]

            if selected:
                _GROQ_MODEL_CANDIDATES_CACHE = list(selected)
                print("    Groq models available to this API key:", ", ".join(selected))
                return list(_GROQ_MODEL_CANDIDATES_CACHE)

    except Exception as exc:
        print("    Groq model discovery warning; using verified fallback IDs:", str(exc)[:250])

    _GROQ_MODEL_CANDIDATES_CACHE = list(preferred)
    return list(_GROQ_MODEL_CANDIDATES_CACHE)


def groq_generate(prompt):
    if not groq_client:
        raise RuntimeError("Groq API key or client is missing")

    metadata_fields = [
        "yt_title",
        "yt_description",
        "yt_tags",
        "tiktok_caption",
        "instagram_caption",
        "fb_caption",
        "popular_captions",
        "hashtags",
        "trending_event",
        "keywords",
        "snapchat_caption",
    ]
    metadata_schema = {
        "type": "object",
        "properties": {field: {"type": "string"} for field in metadata_fields},
        "required": metadata_fields,
        "additionalProperties": False,
    }

    last_exception = None
    for model_candidate in _groq_model_candidates():
        try:
            response = groq_client.chat.completions.create(
                model=model_candidate,
                messages=[
                    {
                        "role": "system",
                        "content": (
                            "You are an expert social-media metadata writer. "
                            "Return valid JSON only. Do not fabricate facts, "
                            "people, news, or current trends."
                        ),
                    },
                    {"role": "user", "content": prompt},
                ],
                temperature=0.5,
                max_tokens=2200,
                response_format={
                    "type": "json_schema",
                    "json_schema": {
                        "name": "social_media_metadata",
                        "strict": True,
                        "schema": metadata_schema,
                    },
                },
            )
            content = response.choices[0].message.content
            return normalize_metadata(extract_json(content)), model_candidate

        except Exception as exc:
            last_exception = exc
            message = str(exc).lower()
            print(f"    Groq strict-JSON model {model_candidate} failed: {str(exc)[:350]}")

            if any(
                marker in message
                for marker in ("model_not_found", "model_decommissioned", "does not exist")
            ):
                continue

            try:
                response = groq_client.chat.completions.create(
                    model=model_candidate,
                    messages=[
                        {
                            "role": "system",
                            "content": (
                                "You are an expert social-media metadata writer. "
                                "Return one valid JSON object only. Do not fabricate "
                                "facts, people, news, or current trends."
                            ),
                        },
                        {"role": "user", "content": prompt},
                    ],
                    temperature=0.5,
                    max_tokens=2200,
                    response_format={"type": "json_object"},
                )
                content = response.choices[0].message.content
                return normalize_metadata(extract_json(content)), model_candidate

            except Exception as fallback_exc:
                last_exception = fallback_exc
                print(f"    Groq JSON-mode fallback failed for {model_candidate}: {str(fallback_exc)[:350]}")
                time.sleep(0.75)

    raise last_exception or RuntimeError("No usable Groq chat model was available.")
"""
replace_section(
    "def groq_generate(prompt):",
    "\n\n# =============================================================================\n# 9. QUALITY SCORING ENGINE",
    groq_function,
    "Groq generation"
)

discord_helpers = r"""
# =============================================================================
# 1A. DISCORD WEBHOOK UPLOAD — ONE VIDEO AT A TIME
# =============================================================================
import mimetypes

DISCORD_WEBHOOK_URL = os.getenv("DISCORD_WEBHOOK_URL", "").strip()
try:
    DISCORD_MAX_FILE_MB = float(os.getenv("DISCORD_MAX_FILE_MB", "20"))
except (TypeError, ValueError):
    DISCORD_MAX_FILE_MB = 20.0
try:
    DISCORD_UPLOAD_TIMEOUT_SECONDS = int(os.getenv("DISCORD_UPLOAD_TIMEOUT_SECONDS", "900"))
except (TypeError, ValueError):
    DISCORD_UPLOAD_TIMEOUT_SECONDS = 900
DISCORD_THREAD_ID = os.getenv("DISCORD_THREAD_ID", "").strip()

if DISCORD_MAX_FILE_MB <= 0:
    DISCORD_MAX_FILE_MB = 20.0
if DISCORD_UPLOAD_TIMEOUT_SECONDS <= 0:
    DISCORD_UPLOAD_TIMEOUT_SECONDS = 900

def validate_discord_webhook():
    response = requests.get(DISCORD_WEBHOOK_URL, timeout=(10, 30))
    if not response.ok:
        raise RuntimeError(f"Discord webhook validation returned HTTP {response.status_code}.")
    info = response.json()
    if not info.get("channel_id"):
        raise RuntimeError("Discord webhook response did not contain a channel_id.")
    return info

DISCORD_WEBHOOK_INFO = {}
DISCORD_READY = False
if DISCORD_WEBHOOK_URL:
    try:
        DISCORD_WEBHOOK_INFO = validate_discord_webhook()
        DISCORD_READY = True
        print("Discord webhook validation: successful")
    except Exception as exc:
        print("Discord webhook warning; continuing without Discord uploads:", str(exc)[:300])
else:
    print("Discord webhook is not configured; metadata processing will continue without uploads.")

def _discord_failure(status, error, size_mb):
    return {
        "status": status,
        "attachment_url": "",
        "message_url": "",
        "message_id": "",
        "channel_id": str(DISCORD_WEBHOOK_INFO.get("channel_id", "")),
        "file_size_mb": round(size_mb, 2),
        "error": str(error)[:1000],
    }

def upload_video_to_discord(video_path, video_name, metadata):
    if not DISCORD_READY:
        return _discord_failure(
            "disabled",
            "Discord webhook was not validated. Check the DISCORD_WEBHOOK_URL secret.",
            0.0,
        )

    path = Path(video_path)
    if not path.is_file():
        return _discord_failure("failed", "Video file does not exist.", 0.0)

    try:
        size_bytes = path.stat().st_size
    except OSError as exc:
        return _discord_failure("failed", f"Could not read video file: {exc}", 0.0)

    size_mb = size_bytes / (1024 * 1024)
    if size_bytes <= 0:
        return _discord_failure("failed", "Video file is empty.", size_mb)

    max_bytes = int(DISCORD_MAX_FILE_MB * 1024 * 1024)
    if size_bytes > max_bytes:
        return _discord_failure(
            "too_large",
            f"Video is {size_mb:.2f} MiB; configured limit is {DISCORD_MAX_FILE_MB:.2f} MiB. "
            "Increase the limit only if Discord allows larger uploads for this account/server.",
            size_mb,
        )

    title = str(metadata.get("yt_title", "") or Path(video_name).stem).strip()
    caption = f"🎬 {title}\n📁 {video_name}"
    short_caption = str(metadata.get("tiktok_caption", "") or "").strip()
    if short_caption:
        caption += "\n\n" + short_caption[:700]
    caption = caption[:1900]

    mime_type = mimetypes.guess_type(str(path))[0] or "application/octet-stream"
    params = {"wait": "true"}
    if DISCORD_THREAD_ID:
        params["thread_id"] = DISCORD_THREAD_ID

    for attempt in range(1, 6):
        try:
            with path.open("rb") as video_file:
                response = requests.post(
                    DISCORD_WEBHOOK_URL,
                    params=params,
                    data={
                        "payload_json": json.dumps(
                            {
                                "content": caption,
                                "allowed_mentions": {"parse": []},
                            },
                            ensure_ascii=False,
                        )
                    },
                    files={"files[0]": (path.name, video_file, mime_type)},
                    timeout=(30, DISCORD_UPLOAD_TIMEOUT_SECONDS),
                )

        except requests.RequestException as exc:
            if attempt == 5:
                return _discord_failure("failed", f"Network error during Discord upload: {exc}", size_mb)
            time.sleep(min(2 ** (attempt - 1), 30))
            continue

        if response.status_code == 429:
            try:
                retry_after = float(
                    response.json().get("retry_after", response.headers.get("Retry-After", 2))
                )
            except (ValueError, TypeError):
                retry_after = 2.0
            if attempt == 5:
                return _discord_failure("failed", "Discord rate limit persisted after five attempts.", size_mb)
            time.sleep(min(max(retry_after, 1.0), 120.0))
            continue

        if 500 <= response.status_code < 600:
            if attempt == 5:
                return _discord_failure("failed", f"Discord server error HTTP {response.status_code}.", size_mb)
            time.sleep(min(2 ** (attempt - 1), 30))
            continue

        if not response.ok:
            try:
                detail = response.json().get("message", "")
            except ValueError:
                detail = response.text[:300]
            return _discord_failure(
                "failed",
                f"Discord HTTP {response.status_code}: {detail or 'upload rejected'}",
                size_mb,
            )

        try:
            message = response.json()
        except ValueError:
            return _discord_failure("failed", "Discord returned no message JSON; link cannot be recorded.", size_mb)

        attachments = message.get("attachments") or []
        if not attachments or not attachments[0].get("url"):
            return _discord_failure("failed", "Discord returned no attachment URL.", size_mb)

        attachment_url = attachments[0]["url"]
        message_id = str(message.get("id", ""))
        channel_id = str(message.get("channel_id") or DISCORD_WEBHOOK_INFO.get("channel_id", ""))
        guild_id = str(DISCORD_WEBHOOK_INFO.get("guild_id") or "")
        message_url = (
            f"https://discord.com/channels/{guild_id}/{channel_id}/{message_id}"
            if guild_id and channel_id and message_id
            else ""
        )
        return {
            "status": "uploaded",
            "attachment_url": attachment_url,
            "message_url": message_url,
            "message_id": message_id,
            "channel_id": channel_id,
            "file_size_mb": round(size_mb, 2),
            "error": "",
        }

    return _discord_failure("failed", "Discord upload failed after five attempts.", size_mb)
"""

marker = "\n\n\n# =============================================================================\n# 2. API KEYS"
if source.count(marker) != 1:
    raise RuntimeError("Safe patch stopped: Discord helper insertion point was not unique.")
source = source.replace(marker, "\n\n\n" + discord_helpers + marker, 1)

replace_once(
    """        except Exception as exc:
            print("  Metadata generation failed completely:", exc)
            failed_count += 1
            continue""",
    """        except Exception as exc:
            print("  Metadata generation failed; using a local fallback and continuing:", str(exc)[:400])
            failed_count += 1
            result = {
                "status": "metadata_failed",
                "ranking_used": False,
                "selected": local_fallback_metadata(video_name),
                "score": "",
                "candidate_scores": "",
                "model_used": "local_filename_fallback",
                "error": str(exc)[:1000],
            }
            metadata = result["selected"]""",
    "metadata failure fallback"
)

replace_once(
    '        # Local workspace only: there is no remote video URL.\n        video_url = "LOCAL_WORKSPACE_ONLY"',
    '        # Filled with the Discord attachment URL after the sequential upload.\n        video_url = ""',
    "video link placeholder"
)

replace_once(
    """        thumbnail_url = create_thumbnail(
            video_path,
            video_name
        )

        generated_at = datetime.now().strftime(""",
    """        thumbnail_url = create_thumbnail(
            video_path,
            video_name
        )

        # Complete this upload before processing the next video.
        discord_result = upload_video_to_discord(video_path, video_name, metadata)
        video_url = discord_result.get("attachment_url", "")

        generated_at = datetime.now().strftime(""",
    "Discord upload call"
)

replace_once(
    """            "pending",
            formatted_time,""",
    """            discord_result.get("status", "not_attempted"),
            formatted_time,""",
    "row upload status"
)

replace_once(
    """            result.get("candidate_scores", ""),
            generated_at
        ]""",
    """            result.get("candidate_scores", ""),
            generated_at,
            result.get("status", ""),
            discord_result.get("status", "not_attempted"),
            discord_result.get("attachment_url", ""),
            discord_result.get("message_url", ""),
            discord_result.get("message_id", ""),
            discord_result.get("channel_id", ""),
            str(discord_result.get("file_size_mb", "")),
            discord_result.get("error", "")
        ]

        if len(row) != len(SHEET_HEADER):
            raise RuntimeError(
                f"Metadata row/header mismatch for {video_name}: {len(row)} values vs {len(SHEET_HEADER)} columns."
            )""",
    "metadata row Discord fields"
)

save_function = """
# Atomic incremental CSV saves protect completed rows if a later video or API call fails.
csv_path = LOCAL_OUTPUT_DIR / "metadata_rows.csv"

def save_metadata_rows(rows):
    temporary_csv_path = csv_path.with_name(csv_path.name + ".tmp")
    with temporary_csv_path.open("w", newline="", encoding="utf-8-sig") as csv_file:
        writer = csv.writer(csv_file)
        writer.writerow(SHEET_HEADER)
        writer.writerows(rows)
    os.replace(temporary_csv_path, csv_path)


def local_fallback_metadata(video_name):
    base = re.sub(r"[_-]+", " ", Path(video_name).stem)
    base = re.sub(r"\\bclip\\s*\\d+\\b", "", base, flags=re.IGNORECASE)
    title = " ".join(base.split())[:90] or "Video"
    return normalize_metadata({
        "yt_title": title,
        "yt_description": (
            "This is a filename-based fallback because the AI metadata service was unavailable. "
            "Review the video and replace this description before publishing."
        ),
        "yt_tags": "video, short video",
        "tiktok_caption": f"Watch: {title}",
        "instagram_caption": f"Watch: {title}",
        "fb_caption": f"Watch: {title}",
        "popular_captions": "",
        "hashtags": "#Shorts #Video",
        "trending_event": "No live trend data verified",
        "keywords": title,
        "snapchat_caption": f"Watch: {title}",
    })

"""
main_marker = "\n\n# =============================================================================\n# 12. MAIN PROCESS"
if source.count(main_marker) != 1:
    raise RuntimeError("Safe patch stopped: main-process insertion point was not unique.")
source = source.replace(main_marker, "\n\n" + save_function + main_marker, 1)

replace_once(
    """        local_rows.append(row)
        existing_names.add(video_name)""",
    """        local_rows.append(row)
        save_metadata_rows(local_rows)
        existing_names.add(video_name)""",
    "incremental CSV save"
)

replace_once(
    """# Save a local copy of newly generated rows and a run summary.
csv_path = LOCAL_OUTPUT_DIR / "metadata_rows.csv"
with csv_path.open("w", newline="", encoding="utf-8-sig") as csv_file:
    writer = csv.writer(csv_file)
    writer.writerow(SHEET_HEADER)
    writer.writerows(local_rows)""",
    """# Final CSV flush (rows were also saved atomically after each video).
save_metadata_rows(local_rows)

discord_status_index = SHEET_HEADER.index("Discord Upload Status")
discord_uploaded_count = sum(
    1 for row in local_rows if row[discord_status_index] == "uploaded"
)
discord_upload_failed_count = sum(
    1 for row in local_rows if row[discord_status_index] != "uploaded"
)""",
    "final CSV save"
)

replace_once(
    '    "source_folder": SOURCE_FOLDER_PATH,',
    '    "source_folder": str(SOURCE_FOLDER_PATH),',
    "JSON-serializable source folder"
)

replace_once(
    """    "local_thumbnail_directory": str(LOCAL_THUMBNAIL_DIR),
    "generated_at": datetime.now(SCHEDULE_TIMEZONE).isoformat(),""",
    """    "local_thumbnail_directory": str(LOCAL_THUMBNAIL_DIR),
    "metadata_csv_path": str(csv_path),
    "metadata_generation_success_count": max(processed_count - failed_count, 0),
    "metadata_generation_failed_count": failed_count,
    "discord_upload_success_count": discord_uploaded_count,
    "discord_upload_failed_count": discord_upload_failed_count,
    "discord_channel_id": str(DISCORD_WEBHOOK_INFO.get("channel_id", "")),
    "discord_file_limit_mib": DISCORD_MAX_FILE_MB,
    "generated_at": datetime.now(SCHEDULE_TIMEZONE).isoformat(),""",
    "run summary diagnostics"
)

replace_once(
    "json.dumps(summary, ensure_ascii=False, indent=2)",
    "json.dumps(summary, ensure_ascii=False, indent=2, default=str)",
    "defensive JSON serialization"
)

replace_once(
    'print("Cloud uploads: disabled (local workspace only)")',
    """print("Metadata generation failures:", failed_count)
print("Discord uploads successful:", discord_uploaded_count)
print("Discord uploads failed, disabled, or too large:", discord_upload_failed_count)
print("Discord attachment and message links saved in:", csv_path)""",
    "final Discord report"
)

compile(source, str(original_path), "exec")
print("Part_4_v2 advanced patch checks passed; original Part_4.ipynb was not modified.")
print("Enabled: live model discovery, JSON-safe AI responses, local fallback metadata, atomic CSV saves, Discord retries and upload diagnostics.")
exec(compile(source, str(original_path), "exec"), globals())
